In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import pandas as pd
import numpy as np
import cv2
import ast
import re


from PIL import Image

import os


This notebook presents several treatments we experimented on our collection, the first ones are purely experimental, but the latest were used to produce actual metadatas. Everything is heavily inspired by https://github.com/taylor-arnold/atelier/tree/main/2026_dv .

These first four code blocks calculate the luminosity of images, using the average color value of image. We then save it to a pandas dataframe 

In [ ]:
folder_name = ""#input here the name of the folder where your images are stored 

In [ ]:
liste_images=[] 
for i in os.listdir(folder_name):
    liste_images.append(f"{folder_name}/{i}")
print(liste_images)

In [ ]:
results = []
for i in liste_images:
    arr = np.asarray(Image.open(i))
    results.append(np.mean(arr) / 255)

In [ ]:
dico={"nom_images": liste_images, "luminosite" : results} 
df = pd.DataFrame(dico)


Now we calculate the chroma of the image, meaning the intensity of the colors of the image, by getting the mean saturation*value element of every pixel in the image. Saturation and value are obtained by converting from RGB to HSV our images. We save both luminosity and chroma in a csv, because the following step can be a bit compute expensive and caused several computer crashes.

In [ ]:
results = []
for index, row in df.iterrows():
    arr = np.asarray(Image.open(row["nom_images"]))
    arr_hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    arr_hsv = arr_hsv.astype(np.float64)
    arr_hsv[:, :, 0] = arr_hsv[:, :, 0] / 179.0
    arr_hsv[:, :, 1] = arr_hsv[:, :, 1] / 255.0
    arr_hsv[:, :, 2] = arr_hsv[:, :, 2] / 255.0
    results.append(np.mean(arr_hsv[:, :, 1] * arr_hsv[:, :, 2]))

df["chroma"] = results 


In [ ]:
df.head(10)
df.to_csv("save_avant_crash.csv")

Now we are going to calculate the colors present on the images. The first two block only load the previously created csv and get the most colorful image by factorizing saturation and value. 

In [ ]:
df = pd.read_csv("save_avant_crash.csv")

In [ ]:
liste_average =[] 
for index, row in df.iterrows():
    arr = np.asarray(Image.open(row["nom_images"]))
    arr_hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    arr_hsv = arr_hsv.astype(np.float64)
    arr_hsv[:, :, 0] = arr_hsv[:, :, 0] / 179.0
    arr_hsv[:, :, 1] = arr_hsv[:, :, 1] / 255.0
    arr_hsv[:, :, 2] = arr_hsv[:, :, 2] / 255.0
    liste_average.append((arr_hsv[:,:,1] * arr_hsv[:,:,2]))
    print(arr_hsv[:,:,1] * arr_hsv[:,:,2])
df["hs"] = liste_average 
df.to_csv("save_avant_crash.csv", sep= ";")

Here we open our "nuances_doublees.csv", which assigns to each to every value between 0 and 1 a corresponding color. As mentionned in the report of the project we enlarged this csv compared to the one used in distant viewing experiments, because the idea is description, so having two "blue" is not too bad, while having for example "flesh" as a color is critical.

In [ ]:
hue = pd.read_csv("nuances_doublees.csv")
hue

Here we get every color present on every picture, and store it in a csv as a list

In [ ]:
liste_cnt =[] 
liste_pourcent =[] 
for index, row in df.iterrows():
    arr = np.asarray(Image.open(row["nom_images"]))

    arr_hsv = cv2.cvtColor(arr, cv2.COLOR_RGB2HSV)
    bins = np.append(0, hue['end'].to_numpy())
    arr_hsv = arr_hsv.astype(np.float64)
    arr_hsv[:, :, 0] = arr_hsv[:, :, 0] / 179.0
    arr_hsv[:, :, 1] = arr_hsv[:, :, 1] / 255.0
    arr_hsv[:, :, 2] = arr_hsv[:, :, 2] / 255.0
    mask = (arr_hsv[:, :, 1] * arr_hsv[:, :, 2]) >0.2  # here we set a threshold, to exclude the pixels were the color is not perceptible, if needed, on oversaturated images of very fainted ones, it can be changed 

    cnt, _ = np.histogram(
        arr_hsv[:, :, 0][mask], bins=bins
    )

    cnt[0] = cnt[0] + cnt[15]
    cnt = cnt[:15]
    liste_cnt.append(cnt.tolist())
    total_pixels = arr_hsv.shape[0] * arr_hsv.shape[1]
    color_percent = np.max(cnt) / total_pixels * 100
    liste_pourcent.append(color_percent)
df["couleurs"] = liste_cnt
df["pourcentage_occupe"] = liste_pourcent
df.to_csv("save_avec_couleurs.csv", sep = ";")

Now we translate the numerical values we obtained to words

In [ ]:
df_color = pd.read_csv("save_avec_couleurs.csv", sep =";")

In [ ]:
liste_couleurs = hue["cnom"] 

In [ ]:
grande_liste =[] 
for key, row in df_color.iterrows() :
    petite_liste =[]
    for i in range (len(ast.literal_eval(row["couleurs"]))) :
        if int(ast.literal_eval(row["couleurs"])[i]) >= 5000:# here we set the minimal amount of pixels for a color to be noted, for a more precise analysis this threshold can be lowered 
            petite_liste.append(liste_couleurs[i])
    grande_liste.append(petite_liste)

  

In [ ]:
grand_liste_2 =[] #simply a small block to merge our separate similar colors (i.e rouge1 and rouge2 become rouge) 
for i in grande_liste :
    petite_liste =[] 
    for z in i :
        petite_liste.append(re.sub(r'[0-9]+', '', z))
    petite_liste= set(petite_liste)
    grand_liste_2.append(petite_liste)
grand_liste_2


In [ ]:
df_color["couleurs_txt_uni_5k"] = grand_liste_2

the following blocks allow us to visualize batches of 4 images with the colors associated to control the result, the variable "compteur" will be set at 0 first, and augment every time we launch the second block, allowing us to check on it to know how many images were controlled.

In [ ]:
compteur = 0  

In [ ]:
a = compteur
b = compteur +1
c = compteur +2
d = compteur +3


fig = plt.figure(figsize=(10, 7))

image1 = cv2.imread(df_color["nom_images"][a])
image2 = cv2.imread(df_color["nom_images"][b])
image3 = cv2.imread(df_color["nom_images"][c])
image4 = cv2.imread(df_color["nom_images"][d])

image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
image2 = cv2.cvtColor(image2, cv2.COLOR_BGR2RGB)
image3 = cv2.cvtColor(image3, cv2.COLOR_BGR2RGB)
image4 = cv2.cvtColor(image4, cv2.COLOR_BGR2RGB)

plt.subplot(2, 2, 1) 
plt.imshow(image1)  
plt.axis('off') 
plt.title(df_color["nom_images"][a] + " " +(",".join(df_color["couleurs_txt_uni_5k"][a]))) 

plt.subplot(2, 2, 2) 
plt.imshow(image2)  
plt.axis('off') 
plt.title(df_color["nom_images"][b] + " " +(",".join(df_color["couleurs_txt_uni_5k"][b])))


plt.subplot(2, 2, 3)  
plt.imshow(image3) 
plt.axis('off')  
plt.title(df_color["nom_images"][c] + " " +(",".join(df_color["couleurs_txt_uni_5k"][c])))  
plt.subplot(2, 2, 4)  
plt.imshow(image4)  
plt.axis('off') 
plt.title(df_color["nom_images"][d] + " " +(",".join(df_color["couleurs_txt_uni_5k"][d])))  
compteur +=4
plt.show()


In [ ]:
df_color.to_csv("csv_avec_couleursV2.csv", sep=";") #and finaly the csv with the main colors mentionned 